# Snapshot Layer User Guide

This notebook is a working guide to the current `Dev/11_Snapshot` layer. It is aimed at two groups:

- accelerator physicists who want to inspect ISIS RCS model behaviour from one entry point;
- GUI and EPICS archiver developers who need stable table-shaped outputs to build interfaces on top of the backend.

The snapshot layer is an orchestration layer. It builds a `MachineState`, runs MAD-X through `MadxModel`, collects tune/orbit/envelope/aperture/correction tables, and returns GUI-ready `pandas.DataFrame` objects. Corrector settings may come from manual entry or EPICS-archiver-shaped tables, and orbit correction suggestions are exposed through snapshot tables.

The key rule is that this repository is read-only with respect to the real machine. A correction result is a suggested model correction, not a controls write.


## Notebook Controls

The default run executes representative examples and keeps heavier full-cycle work optional.

Set `RUN_FULL_CYCLE = True` to build the full-cycle tune programme from the provided 18-point tune data. That section uses every data point defined in `full_cycle_points`, and displays all rows rather than truncating to a preview.

Set `RUN_ORBIT_CORRECTION = False` if MAD-X correction output is not needed during a quick documentation read-through.


In [ ]:
from pathlib import Path
import os
import sys

# Allow execution either from the repository root or from this Dev folder.
if Path.cwd().name != "11_Snapshot":
    candidate = Path.cwd() / "Dev" / "11_Snapshot"
    if candidate.is_dir():
        os.chdir(candidate)

sys.path.insert(0, str(Path.cwd()))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

RUN_APERTURE_EXAMPLE = True
RUN_ORBIT_CORRECTION = True
RUN_FULL_CYCLE = True

GUIDE_OUTPUT_DIR = Path("./snapshot_user_guide_runs")
JAN26_SURVEY_ERROR_TABLE = "../Error_Tables/jan26_survey_corrected.tfs"
LATTICE_FOLDER = "../Lattice_Files/00_Simplified_Lattice"

pd.set_option("display.max_columns", 80)
print(Path.cwd())


In [ ]:
from aperture import (
    plot_aperture_envelope,
    plot_aperture_envelope_with_margin,
    plot_aperture_source_overlay,
    plot_margin,
)
from envelope import EnvelopeInputs, plot_envelope, plot_envelope_comparison, plot_sigma
from orbit_branch import OrbitBranchConfig
from orbit_correction import (
    bpm_measurements_from_twiss,
    enabled_corrector_names,
    normalise_corrector_selection,
    plot_corrector_suggestions,
    plot_orbit_with_bpm,
)
from snapshot import (
    SnapshotConfig,
    SnapshotOrbitCorrectionConfig,
    SnapshotPlotSaveConfig,
    SnapshotSeriesConfig,
    build_full_cycle_snapshot_series,
    build_machine_snapshot,
    build_snapshot_series,
    config_to_dataframe,
    corrector_settings_from_dataframe,
    copy_snapshot_config,
    save_snapshot_plot,
    snapshot_configs_from_timepoint_table,
)
from tune_plots import (
    calculate_harmonic_trim_quad_pattern,
    extract_programmed_trim_quad_pattern,
    plot_beta_variation,
    plot_harmonic_trim_quad_pattern,
    plot_resonance_working_points,
    plot_trim_quad_currents,
)


## 1. Standard Snapshot Setup

A snapshot is the model state at one cycle time. The object you configure is `SnapshotConfig`; the object you get back is `SnapshotResult`.

At a minimum, choose:

- cycle time, in milliseconds;
- requested horizontal and vertical set tunes;
- whether to run envelope and aperture workflows;
- output location for generated MAD-X working files.

The beam state comes from `RCSRamp().state_at(cycle_time_ms)` inside the snapshot layer. The machine state then applies trim quadrupoles, harmonic tune knobs, corrector kicks/currents and optional error table metadata. MAD-X execution remains inside `MadxModel`.


In [ ]:
nominal_config = SnapshotConfig(
    cycle_time_ms=0.0,
    requested_qx=4.31,
    requested_qy=3.83,
    label="nominal injection",
    snapshot_id="guide_nominal_injection",
    output_dir=str(GUIDE_OUTPUT_DIR / "standard"),
    run_envelope=True,
    run_aperture=RUN_APERTURE_EXAMPLE,
    envelope_inputs=EnvelopeInputs(label="nominal injection", sigma_scale=1.0),
)

nominal_snapshot = build_machine_snapshot(nominal_config)

print(nominal_snapshot.available_tables())
nominal_snapshot.to_summary_dataframe()


### Run Directories And Saved Outputs

Every snapshot is written under one timestamped directory inside the configured `output_dir`. A series uses one timestamped parent directory, with all member snapshots beneath it. Saved figures are recorded in `snapshot.table("plot_manifest")` so the GUI can show exactly what was exported.


In [ ]:
display(pd.DataFrame([nominal_snapshot.metadata["run_paths"]]))
assert Path(nominal_snapshot.run_paths.snapshot_dir).is_dir()


### Tables As The GUI Contract

Use `snapshot.table(name)` to retrieve a copy of a table. This is the preferred pattern for GUI and archiver-interface code because it keeps display code decoupled from the physics objects.

Useful tables include:

- `twiss`: MAD-X optics and orbit table;
- `madx_summary`: raw MAD-X summary values;
- `tune_summary`: compact tune/chromaticity values with requested tunes kept distinct from predicted MAD-X tunes;
- `orbit` and `orbit_summary`: GUI-ready orbit values in millimetres;
- `envelope` and `envelope_summary`: beam envelope tables if requested;
- `aperture_aligned` and `aperture_summary`: aperture/margin tables if requested.


In [ ]:
table_sizes = pd.DataFrame(
    [
        {"table": name, "rows": len(nominal_snapshot.table(name)), "columns": len(nominal_snapshot.table(name).columns)}
        for name in nominal_snapshot.available_tables()
    ]
)

display(table_sizes)
display(nominal_snapshot.table("tune_summary"))
display(nominal_snapshot.table("orbit_summary"))


## Corrector Settings From Manual Or EPICS Tables

The GUI can pass corrector settings directly or convert EPICS archiver records into a `SnapshotConfig`. The snapshot keeps both current and MAD-X kick representations available through `corrector_settings` and `corrector_summary` tables.


In [ ]:
archiver_correctors = pd.DataFrame(
    [
        {"cycle_time_ms": 0.0, "corrector": "r0hd1_kick", "plane": "H", "current_A": 0.05},
        {"cycle_time_ms": 0.0, "corrector": "r0vd1_kick", "plane": "V", "current_A": -0.04},
    ]
)
archiver_settings = corrector_settings_from_dataframe(archiver_correctors, cycle_time_ms=0.0)
corrector_snapshot = build_machine_snapshot(
    copy_snapshot_config(
        nominal_config,
        snapshot_id="guide_archiver_correctors",
        label="archiver corrector example",
        output_dir=str(GUIDE_OUTPUT_DIR / "correctors"),
        run_envelope=False,
        run_aperture=False,
        corrector_settings=archiver_settings,
    )
)
display(corrector_snapshot.table("corrector_settings"))
display(corrector_snapshot.table("corrector_summary"))


## 2. Error Tables And Branch Comparisons

The Jan 2026 survey error table is repository-local:

```text
../Error_Tables/jan26_survey_corrected.tfs
```

A snapshot can apply one or more error tables through `error_table_paths`. For a single table the workflow is:

$$
\text{lattice} \rightarrow \text{machine state} \rightarrow \text{USE sequence} \rightarrow \text{SETERR table} \rightarrow \text{TWISS}
$$

`MachineState.error_table_path` records the first table for GUI metadata, while the snapshot metadata records all tables actually applied.


In [ ]:
error_config = copy_snapshot_config(
    nominal_config,
    label="Jan26 survey error injection",
    snapshot_id="guide_jan26_survey_error",
    case="survey_error",
    output_dir=str(GUIDE_OUTPUT_DIR / "error_table"),
    run_aperture=False,
    error_table_paths=[JAN26_SURVEY_ERROR_TABLE],
)

error_snapshot = build_machine_snapshot(error_config)

comparison = pd.concat(
    [
        nominal_snapshot.table("orbit_summary").assign(snapshot="nominal"),
        error_snapshot.table("orbit_summary").assign(snapshot="jan26_survey_error"),
    ],
    ignore_index=True,
)

print(error_snapshot.metadata["applied_error_tables"])
comparison[["snapshot", "max_abs_x_mm", "max_abs_y_mm", "rms_x_mm", "rms_y_mm"]]


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
axes[0].plot(nominal_snapshot.table("orbit")["s"], nominal_snapshot.table("orbit")["x_mm"], label="nominal")
axes[0].plot(error_snapshot.table("orbit")["s"], error_snapshot.table("orbit")["x_mm"], label="Jan26 survey")
axes[0].set_ylabel("x [mm]")
axes[0].legend()

axes[1].plot(nominal_snapshot.table("orbit")["s"], nominal_snapshot.table("orbit")["y_mm"], label="nominal")
axes[1].plot(error_snapshot.table("orbit")["s"], error_snapshot.table("orbit")["y_mm"], label="Jan26 survey")
axes[1].set_xlabel("s [m]")
axes[1].set_ylabel("y [mm]")
axes[1].legend()
fig.tight_layout()
plt.show()


### Orbit Branches

Branches are useful when the GUI needs nominal and variant orbits together. A branch gets its own isolated `MadxModel`, so branch-side errors or correctors do not contaminate the snapshot model.


In [ ]:
branch_config = OrbitBranchConfig(
    name="jan26_survey_branch",
    lattice_folder=LATTICE_FOLDER,
    error_table_paths=[JAN26_SURVEY_ERROR_TABLE],
)

branch_snapshot_config = copy_snapshot_config(
    nominal_config,
    label="nominal with Jan26 branch",
    snapshot_id="guide_nominal_with_branch",
    output_dir=str(GUIDE_OUTPUT_DIR / "branches"),
    run_aperture=False,
    branch_configs=[branch_config],
)
branch_snapshot = build_machine_snapshot(branch_snapshot_config)
branch_snapshot.table("branch_comparison")


## 3. Tune Controls, Harmonics And Full-Cycle Data

The snapshot layer keeps set tunes and predicted MAD-X tunes separate:

$$
(Q_x^{set}, Q_y^{set}) \neq (Q_x^{MAD-X}, Q_y^{MAD-X})
$$

The tune summary also reports chromaticity using the convention already implemented in `tune.py`:

$$
\xi_x = \beta_{rel}\, \frac{dQ_x}{d\delta}, \qquad
\xi_y = \beta_{rel}\, \frac{dQ_y}{d\delta}
$$

Harmonic tune controls are passed as MAD-X global variables, for example `D7COS` and `F8SIN`. The notebook uses the plotting helpers to compare requested harmonic patterns with the programmed lattice pattern.


In [ ]:
harmonics = {"D7COS": 8, "F8SIN": 6}

harmonic_config = copy_snapshot_config(
    nominal_config,
    label="harmonic injection",
    snapshot_id="guide_harmonic_injection",
    output_dir=str(GUIDE_OUTPUT_DIR / "harmonics"),
    run_aperture=False,
    harmonics=harmonics,
    envelope_inputs=EnvelopeInputs(label="harmonic injection", sigma_scale=1.0),
)

harmonic_snapshot = build_machine_snapshot(harmonic_config)

pd.concat(
    [
        nominal_snapshot.table("tune_summary").assign(snapshot="nominal"),
        harmonic_snapshot.table("tune_summary").assign(snapshot="harmonic"),
    ],
    ignore_index=True,
)


In [ ]:
expected_pattern = calculate_harmonic_trim_quad_pattern(
    harmonics,
    brho_Tm=harmonic_snapshot.machine_state.beam_state.brho_Tm,
)
programmed_pattern = extract_programmed_trim_quad_pattern(
    harmonic_snapshot.table("twiss"),
    base_kqtd=harmonic_snapshot.machine_state.kqtd,
    base_kqtf=harmonic_snapshot.machine_state.kqtf,
)
fig, axes = plot_harmonic_trim_quad_pattern(expected_pattern, programmed_pattern, harmonics=harmonics)
plt.show()


In [ ]:
fig, axes = plot_beta_variation(
    nominal_snapshot.table("twiss"),
    harmonic_snapshot.table("twiss"),
    harmonics=harmonics,
)
plt.show()


### Full-Cycle Tune Programme

The following table defines the full-cycle set tune data provided for this project. It is copied from the earlier tune-matching validation notebook and contains all 18 points, including the `-0.1 ms` pre-injection point. The heavy cell below can run all points through MAD-X and display every resulting row.

For GUI/archiver work, the important outputs are:

- `summary`: one row per snapshot;
- `tune_programme`: requested/set tunes plus predicted MAD-X tunes and trim-quad current/K values;
- `working_points`: tune diagram inputs;
- `resonance_lines` and `resonance_proximity`: resonance context for plotting and warnings.


In [ ]:
full_cycle_points = pd.DataFrame(
    {
        "cycle_time_ms": [-0.1, 0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 9.0, 10.0],
        "set_qx": [4.315, 4.270, 4.270, 4.250, 4.235, 4.205, 4.170, 4.190, 4.180, 4.180, 4.180, 4.170, 4.165, 4.165, 4.165, 4.180, 4.180, 4.175],
        "set_qy": [3.820, 3.820, 3.810, 3.805, 3.800, 3.825, 3.680, 3.680, 3.690, 3.700, 3.700, 3.695, 3.695, 3.695, 3.692, 3.690, 3.685, 3.665],
    }
)
assert len(full_cycle_points) == 18
full_cycle_points


In [ ]:
if RUN_FULL_CYCLE:
    full_cycle_base = copy_snapshot_config(
        nominal_config,
        run_envelope=False,
        run_aperture=False,
        output_dir=str(GUIDE_OUTPUT_DIR / "full_cycle"),
        label="full cycle base",
        snapshot_id="guide_full_cycle_base",
    )
    full_cycle = build_full_cycle_snapshot_series(
        full_cycle_points["cycle_time_ms"],
        full_cycle_points["set_qx"],
        full_cycle_points["set_qy"],
        base_config=full_cycle_base,
        label="guide_full_cycle",
    )

    with pd.option_context("display.max_rows", None):
        display(full_cycle.table("tune_programme"))
        display(full_cycle.table("working_points"))
        display(full_cycle.table("resonance_proximity"))

    fig, ax = plot_resonance_working_points(full_cycle.table("tune_programme"))
    plt.show()
    ax = plot_trim_quad_currents(full_cycle.table("tune_programme"))
    plt.show()
else:
    display(Markdown("`RUN_FULL_CYCLE` is `False`. Set it to `True` to run all 18 full-cycle MAD-X points and display every output row."))


## 4. Envelope And Aperture Comparison

Envelope evaluation consumes real TWISS tables and explicit beam assumptions. With geometric RMS emittance the horizontal one-sigma beam size is schematically:

$$
\sigma_x(s) = \sqrt{\beta_x(s)\,\epsilon_x + \left(D_x(s)\,\sigma_p\right)^2}
$$

The plotted envelope bounds use:

$$
x_\pm(s) = x_{orbit}(s) \pm n_\sigma\sigma_x(s)
$$

Aperture margins align MAD-X/source aperture rows to the envelope/orbit table. Zero MAD-X aperture rows from drifts are excluded from limiting-margin calculations.


In [ ]:
ax = plot_envelope(nominal_snapshot.envelope_result, plane="x")
plt.show()
ax = plot_sigma(nominal_snapshot.envelope_result)
plt.show()
ax = plot_envelope_comparison([nominal_snapshot.envelope_result, harmonic_snapshot.envelope_result], plane="x")
plt.show()


In [ ]:
if nominal_snapshot.aperture_result is not None:
    ax = plot_aperture_envelope(nominal_snapshot.aperture_result, plane="x")
    plt.show()
    ax = plot_aperture_envelope_with_margin(nominal_snapshot.aperture_result, plane="y")
    plt.show()
    ax = plot_margin(nominal_snapshot.aperture_result)
    plt.show()
    ax = plot_aperture_source_overlay(
        nominal_snapshot.aperture_result,
        nominal_snapshot.source_aperture_result,
        plane="x",
    )
    plt.show()
    display(nominal_snapshot.table("aperture_summary"))
else:
    display(Markdown("Aperture example skipped because `RUN_APERTURE_EXAMPLE` is `False`."))


## 5. Correcting A Distorted Orbit

This example uses the Jan26 survey error table to create a distorted orbit, then asks the snapshot layer for read-only MAD-X correction suggestions. The GUI-facing outputs are tables: selected BPMs, suggested corrector currents/kicks, before/after monitor summaries, and a per-BPM before/after comparison table.

The correction does not write to the machine. Suggested currents are advisory outputs for operational review.


In [ ]:
distorted_for_correction = error_snapshot

bpm_all_h = bpm_measurements_from_twiss(distorted_for_correction.table("twiss"), plane="H", enabled_default=True)
bpm_all_v = bpm_measurements_from_twiss(distorted_for_correction.table("twiss"), plane="V", enabled_default=True)

bpm_selected_h = bpm_all_h.copy()
bpm_selected_h["enabled"] = bpm_selected_h.index % 2 == 0
bpm_selected_v = bpm_all_v.copy()
bpm_selected_v["enabled"] = bpm_selected_v.index % 2 == 0

h_correctors = normalise_corrector_selection(plane="H")
h_correctors["enabled"] = h_correctors["corrector"].isin(["r0hd1_kick", "r3hd1_kick", "r5hd1_kick", "r9hd1_kick"])
v_correctors = normalise_corrector_selection(plane="V")
v_correctors["enabled"] = v_correctors["corrector"].isin(["r0vd1_kick", "r3vd1_kick", "r5vd1_kick", "r9vd1_kick"])

selection_summary = pd.DataFrame(
    [
        {
            "plane": "H",
            "enabled_bpm": int(bpm_selected_h["enabled"].sum()),
            "enabled_bpm_names": bpm_selected_h.loc[bpm_selected_h["enabled"], "bpm"].tolist(),
            "enabled_correctors": int(h_correctors["enabled"].sum()),
            "enabled_corrector_names": enabled_corrector_names(h_correctors, "H"),
        },
        {
            "plane": "V",
            "enabled_bpm": int(bpm_selected_v["enabled"].sum()),
            "enabled_bpm_names": bpm_selected_v.loc[bpm_selected_v["enabled"], "bpm"].tolist(),
            "enabled_correctors": int(v_correctors["enabled"].sum()),
            "enabled_corrector_names": enabled_corrector_names(v_correctors, "V"),
        },
    ]
)
display(selection_summary)


In [ ]:
if RUN_ORBIT_CORRECTION:
    correction_snapshot = build_machine_snapshot(
        copy_snapshot_config(
            error_config,
            snapshot_id="guide_snapshot_correction",
            label="snapshot correction suggestions",
            output_dir=str(GUIDE_OUTPUT_DIR / "correction"),
            orbit_correction_configs=[
                SnapshotOrbitCorrectionConfig(
                    plane="H",
                    label="horizontal_selected_bpm",
                    bpm_measurements=bpm_selected_h,
                    correctors=h_correctors,
                ),
                SnapshotOrbitCorrectionConfig(
                    plane="V",
                    label="vertical_selected_bpm",
                    bpm_measurements=bpm_selected_v,
                    correctors=v_correctors,
                ),
            ],
        )
    )
    display(correction_snapshot.table("orbit_correction_summary"))
    display(correction_snapshot.table("orbit_correction_bpm_comparison").head(12))
    display(correction_snapshot.table("orbit_correction_correctors"))
else:
    display(Markdown("Orbit correction skipped because `RUN_ORBIT_CORRECTION` is `False`."))


In [ ]:
if RUN_ORBIT_CORRECTION:
    h_correction = next(result.result for result in correction_snapshot.orbit_correction_results if result.plane == "H")
    v_correction = next(result.result for result in correction_snapshot.orbit_correction_results if result.plane == "V")

    fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)
    plot_orbit_with_bpm(h_correction.measured_twiss_df, h_correction.bpm_measurements, plane="H", ax=axes[0], label="Measured fitted orbit", title="Horizontal corrected orbit suggestion")
    plot_orbit_with_bpm(h_correction.corrected_twiss_df, h_correction.bpm_measurements, plane="H", ax=axes[0], label="Corrected orbit suggestion", orbit_kwargs={"linestyle": "--"})
    plot_orbit_with_bpm(v_correction.measured_twiss_df, v_correction.bpm_measurements, plane="V", ax=axes[1], label="Measured fitted orbit", title="Vertical corrected orbit suggestion")
    plot_orbit_with_bpm(v_correction.corrected_twiss_df, v_correction.bpm_measurements, plane="V", ax=axes[1], label="Corrected orbit suggestion", orbit_kwargs={"linestyle": "--"})
    fig.tight_layout()
    save_snapshot_plot(correction_snapshot, fig, plot_name="corrected_orbit_suggestions", aspect="correction")
    plt.show()

    fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=False)
    plot_corrector_suggestions(h_correction.correctors, ax=axes[0], value="delta_current_A", title="Horizontal selected correctors")
    plot_corrector_suggestions(v_correction.correctors, ax=axes[1], value="delta_current_A", title="Vertical selected correctors")
    fig.tight_layout()
    save_snapshot_plot(correction_snapshot, fig, plot_name="corrector_current_suggestions", aspect="correction")
    plt.show()

    display(correction_snapshot.table("plot_manifest"))


## 6. Notes For GUI And EPICS Archiver Layers

Recommended interface pattern:

1. A GUI form builds a `SnapshotConfig` or a copied variant from an existing config.
2. The backend calls `build_machine_snapshot(config)` or `build_snapshot_series(series_config)`.
3. GUI widgets consume `result.table(name)` DataFrames and `result.metadata` dictionaries.
4. Archiver integration should translate EPICS records into config fields or measurement tables, then store the returned DataFrames and metadata with a timestamp.

Good GUI panels to build from this layer:

- machine-state summary: `snapshot.machine_state.summary_dict()`;
- tune summary: `snapshot.table("tune_summary")`;
- orbit plot and RMS cards: `snapshot.table("orbit")`, `snapshot.table("orbit_summary")`;
- envelope/aperture tabs: `snapshot.table("envelope")`, `snapshot.table("aperture_aligned")`, `snapshot.table("aperture_summary")`;
- correction suggestion panel: selected BPM table, selected corrector table, MAD-X correction table, and current suggestions.

Do not write generated correction currents to controls from this layer. Treat them as model suggestions requiring operational review.


In [ ]:
gui_contract = {
    "snapshot_id": nominal_snapshot.snapshot_id,
    "label": nominal_snapshot.label,
    "tables": nominal_snapshot.available_tables(),
    "metadata_keys": sorted(nominal_snapshot.metadata.keys()),
    "machine_state_keys": sorted(nominal_snapshot.machine_state.summary_dict().keys()),
}

gui_contract


## 7. Minimal Recipes

### One nominal snapshot

```python
config = SnapshotConfig(
    cycle_time_ms=0.0,
    requested_qx=4.31,
    requested_qy=3.83,
    output_dir="./snapshot_runs/minimal",
)
result = build_machine_snapshot(config)
twiss = result.table("twiss")
```

### Copy a config and add an error table

```python
error_config = copy_snapshot_config(
    config,
    error_table_paths=["../Error_Tables/jan26_survey_corrected.tfs"],
)
error_result = build_machine_snapshot(error_config)
```

### Full-cycle tune programme with per-point settings

```python
full_cycle = build_full_cycle_snapshot_series(
    cycle_times_ms,
    qx_values,
    qy_values,
    base_config=copy_snapshot_config(config, run_envelope=False, run_aperture=False),
    point_overrides=[{"corrector_settings": settings_for_this_time}, ...],
    output_dir="./snapshot_runs/full_cycle",
)
tune_programme = full_cycle.table("tune_programme")
```

### Selected BPM/corrector correction suggestion

```python
correction_config = copy_snapshot_config(
    distorted_config,
    orbit_correction_configs=[
        SnapshotOrbitCorrectionConfig(
            plane="H",
            bpm_measurements=selected_bpm_table,
            correctors=selected_corrector_table,
        )
    ],
)
correction_result = build_machine_snapshot(correction_config)
bpm_before_after = correction_result.table("orbit_correction_bpm_comparison")
current_suggestions = correction_result.table("orbit_correction_correctors")
```
